# Pre-processing: dummy flood-depth map

A **stand-in for a real hydrodynamic flood model**, so the rest of the pipeline can be exercised
end-to-end. We build a flood-depth raster with a simple **bathtub** inundation model over a real
global DEM — **Copernicus GLO-30** (~30 m, public, no API key) — and fall back to a synthetic
terrain surface if the DEM can't be fetched (e.g. offline).

The output is a single-band GeoTIFF of flood depth in metres (positive = flooded, 0 = dry) in a
projected CRS. It's consumed by `4_run_model.ipynb`; the model reprojects/clips it onto the
population grid automatically, so **no alignment step is needed here**.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rioxarray  # registers the .rio accessor
from rasterio.transform import from_origin
from rasterio.warp import transform_bounds

from d_health.io import from_numpy

## 2. Configure

Same Paramaribo (Suriname) bounding box as the population notebook, so the flood footprint lines up
with the population raster. The flood map is built in **UTM 21N (EPSG:32621)** — flood maps are
conventionally in a projected, metric CRS. `water_level_m` is the **absolute** water-surface
elevation (relative to the DEM datum, ≈ mean sea level): every cell whose terrain is below it is
flooded. Raise it for a larger / deeper flood.

In [ ]:
# Area of interest in EPSG:4326 (xmin, ymin, xmax, ymax)
aoi_4326 = (-55.27, 5.78, -55.10, 5.93)

flood_crs = "EPSG:32621"   # UTM zone 21N — Suriname (metric)
target_res = 30.0          # output resolution in metres (matches GLO-30)
water_level_m = 3.0        # absolute water-surface elevation (m), relative to the
                           # DEM datum (≈ mean sea level): flood every cell below it

output_dir = Path("./data")
output_dir.mkdir(parents=True, exist_ok=True)
flood_path = output_dir / "flood_depth_paramaribo.tif"
flood_path

## 3. Fetch a global DEM (Copernicus GLO-30) — with synthetic fallback

`fetch_dem` builds the GLO-30 tile name(s) covering the AOI, opens the public COG(s) straight over
HTTPS with GDAL's `/vsicurl/` (no download, no API key), mosaics if needed, clips to the AOI, and
reprojects to the target CRS/resolution. If anything fails (offline, network blocked), we fall back
to `synthetic_dem` — a smooth procedural surface — so the notebook always produces a flood map.

In [ ]:
def cop_glo30_tile(lat: float, lon: float) -> str:
    """Copernicus GLO-30 COG tile id covering the 1° cell containing (lat, lon)."""
    ns = "N" if lat >= 0 else "S"
    ew = "E" if lon >= 0 else "W"
    return (
        f"Copernicus_DSM_COG_10_{ns}{abs(int(np.floor(lat))):02d}_00_"
        f"{ew}{abs(int(np.floor(lon))):03d}_00_DEM"
    )


def fetch_dem(bounds_4326, dst_crs, res):
    """Fetch + clip + reproject the Copernicus GLO-30 DEM for an AOI (EPSG:4326 bounds)."""
    import os
    from rioxarray.merge import merge_arrays

    # Speed/robustness hints for reading remote COGs.
    os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
    os.environ.setdefault("GDAL_HTTP_TIMEOUT", "30")
    os.environ.setdefault("CPL_VSIL_CURL_USE_HEAD", "NO")

    xmin, ymin, xmax, ymax = bounds_4326
    tiles = sorted({
        cop_glo30_tile(lat, lon)
        for lat in range(int(np.floor(ymin)), int(np.floor(ymax)) + 1)
        for lon in range(int(np.floor(xmin)), int(np.floor(xmax)) + 1)
    })
    arrs = [
        rioxarray.open_rasterio(
            f"/vsicurl/https://copernicus-dem-30m.s3.amazonaws.com/{t}/{t}.tif",
            masked=True,
        )
        for t in tiles
    ]
    dem = arrs[0] if len(arrs) == 1 else merge_arrays(arrs)
    dem = dem.rio.clip_box(xmin, ymin, xmax, ymax)
    dem = dem.rio.reproject(dst_crs, resolution=res)
    if "band" in dem.dims:
        dem = dem.squeeze("band", drop=True)
    return dem.rename("elevation")


def synthetic_dem(bounds_4326, dst_crs, res):
    """Smooth procedural terrain on the AOI grid, used when the DEM can't be fetched."""
    xmin, ymin, xmax, ymax = transform_bounds("EPSG:4326", dst_crs, *bounds_4326)
    xmin, ymin = np.floor(xmin / res) * res, np.floor(ymin / res) * res
    xmax, ymax = np.ceil(xmax / res) * res, np.ceil(ymax / res) * res
    width = int(round((xmax - xmin) / res))
    height = int(round((ymax - ymin) / res))

    yy, xx = np.mgrid[0:height, 0:width]
    xn, yn = xx / width, yy / height
    elev = 8.0 + 12.0 * xn + 6.0 * yn                      # gentle tilt, ~8..26 m
    for cx, cy, amp, s in [(0.35, 0.45, -7.0, 0.22), (0.70, 0.65, 9.0, 0.18)]:
        elev += amp * np.exp(-(((xn - cx) / s) ** 2 + ((yn - cy) / s) ** 2))

    transform = from_origin(xmin, ymax, res, res)
    return from_numpy(elev.astype(np.float32), transform, dst_crs, name="elevation")


try:
    elevation = fetch_dem(aoi_4326, flood_crs, target_res)
    print(f"DEM: Copernicus GLO-30 {tuple(elevation.shape)} in {elevation.rio.crs}")
except Exception as exc:  # offline / blocked / missing tile
    print(f"DEM fetch failed ({type(exc).__name__}: {exc}); using synthetic terrain.")
    elevation = synthetic_dem(aoi_4326, flood_crs, target_res)
    print(f"DEM: synthetic {tuple(elevation.shape)} in {elevation.rio.crs}")

## 4. Bathtub flood

Fill the terrain like a bathtub up to a fixed **absolute** water level (`water_level_m`, relative to
zero / the DEM datum): `depth = max(0, water_level - elevation)`. Cells below a small noise floor are
set to dry (0). This is deliberately simple (no hydrological connectivity) — enough to drive the demo.

In [ ]:
def bathtub(elev_da, water_level, noise=0.05):
    """Flood everything below an absolute `water_level` (m): depth = max(0, water_level - elevation)."""
    elev = elev_da.values.astype(np.float32)
    depth = np.clip(water_level - elev, 0.0, None).astype(np.float32)
    depth[depth < noise] = 0.0
    depth[np.isnan(elev)] = 0.0
    da = elev_da.copy(data=depth).rename("flood_depth")
    da = da.assign_attrs(long_name="flood_depth_m", units="m")
    da = da.rio.write_nodata(0)
    return da


flood = bathtub(elevation, water_level_m)

elev_vals = elevation.values
print(f"elevation  : {np.nanmin(elev_vals):.1f} .. {np.nanmax(elev_vals):.1f} m")
print(f"water level: {water_level_m:.2f} m (absolute)")
print(f"max depth  : {flood.values.max():.2f} m")
flooded = int((flood.values > 0).sum())
print(f"flooded    : {flooded:,} / {flood.size:,} cells ({100 * flooded / flood.size:.0f}%)")

## 5. Write the flood-depth GeoTIFF

In [ ]:
flood.rio.to_raster(flood_path, dtype="float32", compress="deflate")

import rasterio
with rasterio.open(flood_path) as src:
    print(f"Wrote {flood_path}")
    print(f"  CRS: {src.crs}, resolution: {src.res[0]:.0f} m, shape: {src.shape}, nodata: {src.nodata}")

## 6. Visualise

Elevation on the left, the bathtub flood depth on the right (same metric extent).

In [ ]:
left, bottom, right, top = flood.rio.bounds()
extent = (left, right, bottom, top)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

im0 = axes[0].imshow(elev_vals, cmap="terrain", extent=extent)
axes[0].set_title("Elevation (m)")
fig.colorbar(im0, ax=axes[0], shrink=0.7, label="m")

depth_masked = np.ma.masked_where(flood.values <= 0, flood.values)
im1 = axes[1].imshow(elev_vals, cmap="Greys_r", extent=extent, alpha=0.6)
im2 = axes[1].imshow(depth_masked, cmap="Blues", extent=extent, vmin=0)
axes[1].set_title(f"Flood depth (m) — bathtub @ {water_level_m:.1f} m")
fig.colorbar(im2, ax=axes[1], shrink=0.7, label="depth (m)")

for ax in axes:
    ax.set_xlabel("easting (m)")
    ax.set_ylabel("northing (m)")
fig.suptitle("Dummy flood-depth map — Paramaribo")
plt.show()